# landcoverai-finetune — can a Polish land-cover model give us `track`?

**Status:** `exploring`

LandCover.ai is the closest annotated dataset that exists to this problem:
**rural Poland**, 0.25 m (33 orthophotos) and 0.5 m (8), 216.27 km², manually
fine-annotated, split into 10,674 tiles of 512x512 (7,470 train / 1,602 val /
1,602 test). Classes: **building, woodland, water, road** — four of our five,
and it is the **only candidate with a road class**, which is the class nothing
else supplies.

Class areas: woodland 72.02 km², water 13.15 km², road 3.50 km², building
1.85 km². Roads are 1.6% of the labelled area — the class we most need is the
rarest, and that has to be handled in the loss, not discovered afterwards.

**Licence: CC-BY-NC-SA 4.0.** Non-commercial *and* share-alike. Fine for offline
label generation and academic publication; the share-alike may attach to weights
trained on it. Not a blocker, but it must be in the dataset card.

### This probe is gated on `flair-landcover-transfer`

Both ask the same underlying question — *does a Western-European land-cover model
transfer to Ukrainian Esri tiles?* — and FLAIR answers it with no training at
all. **If FLAIR fails on `building` and `water` at both GSDs, stop here.**

### Compute reality, stated before starting

This machine is an **Intel i7-9750H, 6 cores, 16 GB, no CUDA, no MPS**. A full
ResNet34-UNet run over 7,470 tiles at 512x512 is days, not hours. Stage 0 below
**measures** seconds/batch on this hardware and prints the extrapolated epoch
time, so the decision to train is made against a number rather than a hope.
Training is opt-in (`train=False` by default).

### What would make this `failed`

- `road` IoU on LandCover.ai's own test split is so low the class was never
  learned — a Polish-data failure, before Ukraine is involved.
- The model transfers on building/water but `road` collapses over Ukrainian
  tiles — paved-road bias, the failure the literature already reports for
  unpaved tracks.
- Predicted tracks disagree across two dates of the same ground.

Run from `src/` with the `uavloc` env.
`papermill pocs/landcoverai-finetune/probe.ipynb pocs/landcoverai-finetune/runs/probe-$(date +%F).ipynb -p train True`

In [ ]:
# papermill parameters
SLUG = "landcoverai-finetune"
n = 9                  # Ukrainian locations for the transfer eval
train = False          # stage 1 is opt-in — measure the budget first
crop = 256             # smaller than LandCover.ai's 512: 4x cheaper per sample
batch = 8
epochs = 6
max_train_tiles = 2000 # subset; a probe does not need the full 7,470
lr = 3e-4
encoder = "resnet34"
data_url = "https://landcover.ai.linuxpolska.com/download/landcover.ai.v1.zip"

## Setup

In [ ]:
import gzip, io, json, sys, time, urllib.request, warnings
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

assert Path("datasets").is_dir(), "run from src/ — cwd is " + str(Path.cwd())

sys.path.insert(0, str(Path.cwd() / "pocs" / "map-derived-labels"))
import align, osm                              # noqa: E402
from tiles import TileGrid                     # noqa: E402

ROOT = Path("data/derived/ua-village-multidate")
HERE = Path("pocs") / SLUG
FIGS = HERE / "figures"
FIGS.mkdir(parents=True, exist_ok=True)


def locations():
    return json.loads((ROOT / "index.json").read_text())["built"]


def load_location(name, date="current"):
    """(image, grid, meta) at full resolution — same path build.py rasterised through."""
    meta = json.loads((ROOT / name / "meta.json").read_text())
    img = Image.open(ROOT / name / f"{date}.jpg").convert("RGB")
    return img, TileGrid(**meta["grid"]), meta


def osm_mask(name, cls):
    """One class from the dataset's own label.png, shifted by its measured offset.

    The build recorded `label_offset` per location. Charging the dataset's known
    2.84 m label error to the thing under test would be unfair, so undo it here.
    """
    meta = json.loads((ROOT / name / "meta.json").read_text())
    lab = np.array(Image.open(ROOT / name / "label.png"))
    m = lab == osm.CLASSES[cls]
    off = meta.get("label_offset") or {}
    dx, dy = int(off.get("dx_px", 0)), int(off.get("dy_px", 0))
    if dx or dy:
        m = np.roll(np.roll(m, dy, axis=0), dx, axis=1)
    return m


def iou(a, b):
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else float("nan")


def null_iou(pred, truth, k=20, seed=0):
    """Same mask, position destroyed. Anything not beating this is an artifact
    of how much ground got painted (bug-016)."""
    rng = np.random.default_rng(seed)
    h, w = pred.shape
    return float(np.median([
        iou(np.roll(np.roll(pred, rng.integers(h // 8, h - h // 8), 0),
                    rng.integers(w // 8, w - w // 8), 1), truth)
        for _ in range(k)
    ]))

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

torch.set_num_threads(6)
print("torch", torch.__version__, "| smp", smp.__version__)

DATA = Path("data/derived/landcoverai")
DATA.mkdir(parents=True, exist_ok=True)

# LandCover.ai mask values -> our names. Index 0 is unlabelled/other.
LCAI = {0: "background", 1: "building", 2: "woodland", 3: "water", 4: "road"}
OURS = ["background", "building", "track", "woody", "water"]
LCAI_TO_OURS = {0: 0, 1: 1, 2: 3, 3: 4, 4: 2}

## Stage 0a — get the data

~1.4-2.9 GB zipped. **29 GB free on this disk**, and extracted tiles add several
more, so check before unpacking. `split.py` ships with the archive and cuts the
41 orthophotos into 512x512 tiles listed by `train.txt` / `val.txt` / `test.txt`.

In [ ]:
import shutil

free_gb = shutil.disk_usage(DATA).free / 2**30
print(f"free disk: {free_gb:.1f} GB")
assert free_gb > 12, "not enough room to unpack LandCover.ai — free space first"

zip_path = DATA / "landcover.ai.v1.zip"
if not zip_path.exists():
    print("downloading — this is the slow part")
    urllib.request.urlretrieve(data_url, zip_path)
print(f"{zip_path.name}: {zip_path.stat().st_size / 2**30:.2f} GB")

if not (DATA / "images").is_dir():
    import zipfile
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
print(sorted(p.name for p in DATA.iterdir()))

In [ ]:
# split.py ships in the archive and writes output/<tile>.jpg + <tile>_m.png
if not (DATA / "output").is_dir():
    import subprocess
    print("tiling — a few minutes")
    subprocess.run([sys.executable, "split.py"], cwd=DATA, check=True)

def split_list(which):
    return [l.strip() for l in (DATA / f"{which}.txt").read_text().split("\n") if l.strip()]

splits = {w: split_list(w) for w in ("train", "val", "test")}
for w, v in splits.items():
    print(f"{w:>6s}: {len(v):>6,} tiles")

## Stage 0b — the compute budget, measured

One forward+backward on **this** CPU, timed. Everything downstream is arithmetic
from this number. If the extrapolated run is longer than a night, the honest move
is to shrink the probe or rent a GPU — not to start it and hope.

In [ ]:
def build_model():
    m = smp.Unet(encoder_name=encoder, encoder_weights="imagenet",
                 in_channels=3, classes=len(OURS))
    return m


probe_model = build_model()
opt = torch.optim.AdamW(probe_model.parameters(), lr=lr)
x = torch.randn(batch, 3, crop, crop)
y = torch.randint(0, len(OURS), (batch, crop, crop))
lossfn = nn.CrossEntropyLoss()

probe_model.train()
t0 = time.time()
for _ in range(3):
    opt.zero_grad()
    lossfn(probe_model(x), y).backward()
    opt.step()
per_batch = (time.time() - t0) / 3

n_tiles = min(max_train_tiles, len(splits["train"]))
per_epoch = per_batch * n_tiles / batch
print(f"seconds/batch (b={batch}, {crop}px): {per_batch:.2f}")
print(f"tiles/epoch: {n_tiles:,}  ->  {per_epoch/60:.1f} min/epoch")
print(f"{epochs} epochs: {per_epoch*epochs/3600:.1f} hours")
print(f"\nfull dataset at 512px would be ~{per_batch*4*len(splits['train'])/batch*epochs/3600:.0f} h")
print("\n-> set train=True only if the number above is acceptable")

## Stage 1 — fine-tune (opt-in)

From an ImageNet encoder, 256 px crops, a subset, few epochs. This is a **probe
model**, not a contender for LandCover.ai's 85.6% mIoU benchmark — the question
is whether `road` is learnable at all and whether it survives the trip to
Ukraine, not whether we can match the paper.

`road` is 1.6% of labelled area, so the loss is class-weighted. Without that the
model predicts "not road" everywhere and reports a fine accuracy — the same
shape of mistake as `bug-016`.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class LandCoverAI(Dataset):
    def __init__(self, names, crop=256, augment=False):
        self.names, self.crop, self.augment = names, crop, augment

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        stem = self.names[i]
        img = np.array(Image.open(DATA / "output" / f"{stem}.jpg").convert("RGB"))
        msk = np.array(Image.open(DATA / "output" / f"{stem}_m.png"))
        # split.py writes masks with cv2.imread() and no grayscale flag, so every
        # mask PNG is 3-channel with the three channels identical. Feeding that
        # to CrossEntropyLoss gives a (H, W, 3) target and a shape error.
        if msk.ndim == 3:
            msk = msk[..., 0]
        if self.crop < img.shape[0]:
            top = np.random.randint(0, img.shape[0] - self.crop + 1)
            left = np.random.randint(0, img.shape[1] - self.crop + 1)
            img = img[top:top + self.crop, left:left + self.crop]
            msk = msk[top:top + self.crop, left:left + self.crop]
        if self.augment and np.random.rand() < 0.5:
            img, msk = img[:, ::-1].copy(), msk[:, ::-1].copy()
        out = np.zeros_like(msk)
        for src, dst in LCAI_TO_OURS.items():
            out[msk == src] = dst
        x = (img.astype(np.float32) / 255.0 - 0.5) / 0.25
        return torch.from_numpy(x.transpose(2, 0, 1)), torch.from_numpy(out).long()


model = build_model()
ckpt = HERE / "model.pt"

if train:
    # num_workers=0 deliberately. macOS multiprocessing uses *spawn*, and a class
    # defined in a notebook cell cannot be re-imported by the child, so any
    # num_workers>0 dies with "Can't pickle <class 'LandCoverAI'>". It would fail
    # identically under papermill. In-process loading also avoids contending with
    # torch's 6 intra-op threads, which already saturate this CPU.
    tr = DataLoader(LandCoverAI(splits["train"][:max_train_tiles], crop, True),
                    batch_size=batch, shuffle=True, num_workers=0, drop_last=True)
    va = DataLoader(LandCoverAI(splits["val"][:400], crop), batch_size=batch, num_workers=0)

    # Median-frequency balancing, from pixel counts MEASURED over 150 random
    # train tiles (2026-09-24) rather than guessed:
    #   background 63.50%  building 0.87%  track 1.52%  woody 26.97%  water 7.15%
    # w_c = median(f) / f_c. The earlier hand-picked vector had track weighted
    # above building while building is the rarer class -- exactly the guess-before-
    # measuring mistake recorded for MIN_BUILDING_PX (bug-012).
    w = torch.tensor([0.113, 8.22, 4.70, 0.265, 1.0])
    lossfn = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * len(tr))

    for ep in range(epochs):
        model.train()
        t0, run = time.time(), 0.0
        for i, (x, y) in enumerate(tr):
            opt.zero_grad()
            loss = lossfn(model(x), y)
            loss.backward()
            opt.step()
            sched.step()
            run += loss.item()
            if i % 25 == 0:
                print(f"  ep{ep} {i}/{len(tr)} loss {run/(i+1):.4f} "
                      f"({(time.time()-t0)/60:.1f} min)", flush=True)

        model.eval()
        inter = np.zeros(len(OURS))
        union = np.zeros(len(OURS))
        with torch.no_grad():
            for x, y in va:
                p = model(x).argmax(1)
                for c in range(len(OURS)):
                    inter[c] += ((p == c) & (y == c)).sum().item()
                    union[c] += ((p == c) | (y == c)).sum().item()
        ious = inter / np.maximum(union, 1)
        print(f"ep{ep} val IoU " + "  ".join(f"{c}={v:.3f}" for c, v in zip(OURS, ious)))
        torch.save(model.state_dict(), ckpt)
elif ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    print("loaded", ckpt)
else:
    print("no checkpoint and train=False — stage 2 will be skipped")
model.eval()

## Stage 2 — does it cross the border?

Same control as every other probe: OSM `building` and `water` over the Ukrainian
tiles, offset-corrected, with a null baseline. `track` has **no** control — OSM
does not map the tracks we care about, which is the whole reason for this class —
so it is scored two other ways:

- **cross-date consistency**, which needs no truth
- **eyes, at full resolution**, which is what `bug-014` cost us last time

A `track` number without one of those is not evidence.

In [ ]:
@torch.no_grad()
def predict_ua(img, mpp, target_mpp=0.25):
    """LandCover.ai is 0.25 m/px; our tiles are 0.386. Resample to the training
    GSD, tile, predict, map back. Compare by ground scale, never by pixel count
    (the mistake recorded in pocs/drone-frame-boundaries)."""
    w0, h0 = img.size
    scale = mpp / target_mpp
    im = img.resize((int(w0 * scale), int(h0 * scale)), Image.BICUBIC)
    arr = (np.asarray(im, np.float32) / 255.0 - 0.5) / 0.25
    h, w = arr.shape[:2]
    logits = np.zeros((len(OURS), h, w), np.float32)
    count = np.zeros((h, w), np.float32)
    step = crop
    for top in range(0, h, step):
        for left in range(0, w, step):
            t, l = min(top, max(h - crop, 0)), min(left, max(w - crop, 0))
            tile = arr[t:t + crop, l:l + crop]
            ph, pw = tile.shape[:2]
            if ph < crop or pw < crop:
                tile = np.pad(tile, ((0, crop - ph), (0, crop - pw), (0, 0)))
            out = model(torch.from_numpy(tile.transpose(2, 0, 1))[None])[0].numpy()
            logits[:, t:t + ph, l:l + pw] += out[:, :ph, :pw]
            count[t:t + ph, l:l + pw] += 1
    logits /= np.maximum(count, 1)
    pred = logits.argmax(0).astype(np.uint8)
    return np.array(Image.fromarray(pred).resize((w0, h0), Image.NEAREST))


names = locations()[:n]
transfer, preds = [], {}
if train or ckpt.exists():
    for name in names:
        img, _, meta = load_location(name)
        p = predict_ua(img, meta["mpp"])
        preds[name] = p
        row = {"location": name}
        for i, cls in enumerate(OURS):
            row[cls] = {"pred_frac": float((p == i).mean())}
        for cls, osm_cls in (("building", "building"), ("water", "water"), ("woody", "wood")):
            m, t = p == OURS.index(cls), osm_mask(name, osm_cls)
            row[cls].update(iou=iou(m, t), null_iou=null_iou(m, t),
                            recall=float((m & t).sum() / t.sum()) if t.sum() else float("nan"))
        transfer.append(row)
        print(f"{name:30s} building IoU {row['building']['iou']:.3f} "
              f"(null {row['building']['null_iou']:.3f})  "
              f"track claims {row['track']['pred_frac']:.2%}")
    (HERE / "results-transfer.json").write_text(json.dumps(transfer, indent=1))

## Numbers

In [ ]:
import statistics as st

if transfer:
    print(f"{'class':>10s} {'IoU':>7s} {'null':>7s} {'recall':>7s} {'claims':>8s}")
    for cls in OURS:
        rows = [r[cls] for r in transfer]
        if "iou" in rows[0]:
            m = lambda k: st.median([r[k] for r in rows if not np.isnan(r[k])])  # noqa: E731
            print(f"{cls:>10s} {m('iou'):>7.3f} {m('null_iou'):>7.3f} "
                  f"{m('recall'):>7.3f} {m('pred_frac'):>8.2%}")
        else:
            print(f"{cls:>10s} {'—':>7s} {'—':>7s} {'—':>7s} "
                  f"{st.median([r['pred_frac'] for r in rows]):>8.2%}   (no control)")

## Look at it

`track` has no control, so this figure **is** the evidence for that class.
Full resolution. The question to answer by eye: are the predicted tracks on the
actual lanes and farm roads, or are they field edges and shadows?

In [ ]:
palette = {"building": (1, 0, 0), "track": (1, 0.85, 0),
           "woody": (0, 0.8, 0), "water": (0, 0.4, 1)}
show = [n_ for n_ in names[:3] if n_ in preds]
if show:
    fig, axes = plt.subplots(len(show), 3, figsize=(16, 5.2 * len(show)))
    axes = np.atleast_2d(axes)
    for row, name in enumerate(show):
        img, _, _ = load_location(name)
        arr = np.array(img)
        p = preds[name]
        over = np.zeros((*arr.shape[:2], 3))
        for cls, colour in palette.items():
            m = p == OURS.index(cls)
            for c in range(3):
                over[..., c] = np.where(m, colour[c], over[..., c])
        axes[row, 0].imshow(arr)
        axes[row, 0].set_title(f"{name}\nimagery", fontsize=8)
        axes[row, 1].imshow(arr)
        axes[row, 1].imshow(over, alpha=0.5)
        axes[row, 1].set_title("prediction (yellow = track)", fontsize=8)

        trk = p == OURS.index("track")
        ys, xs = np.nonzero(trk)
        if len(ys):
            cy, cx = int(np.median(ys)), int(np.median(xs))
            h = 320
            sl = (slice(max(0, cy - h), cy + h), slice(max(0, cx - h), cx + h))
            axes[row, 2].imshow(arr[sl])
            axes[row, 2].imshow(over[sl], alpha=0.5)
        axes[row, 2].set_title("native-resolution zoom — is that a track?", fontsize=8)
        for a in axes[row]:
            a.axis("off")
    plt.tight_layout()
    plt.savefig(FIGS / "landcoverai-transfer.png", dpi=130, bbox_inches="tight")
    plt.show()

## Verdict

`worked` / `failed` / `parked`, two or three lines, then copy into `notes.md`.

Report **two** verdicts, they are separate findings:
1. Did `road` learn on LandCover.ai's own Polish test split?
2. Did it survive the move to Ukrainian Esri tiles?

A yes to (1) and no to (2) is the paved-road-bias result and it is worth writing
up properly — it would close the `track`-from-a-model route.

### What is *not* established